[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/projects/04_predictive-maintenance.ipynb)

        *Industry Project 4 of 8 · ML Engineer · Intermediate–Advanced · about 8–10 hours*

        # Predictive Maintenance from Sensor Data

        **The situation.** A packaging plant runs 40 machines. When one breaks unexpectedly the line stops for a day, costing about **₹2,00,000**. An inspection costs **₹20,000** and fixes the problem if done in time. Each machine reports daily temperature, vibration and pressure.

The plant manager asks: *"Can you warn us a few days before a machine fails, without sending the crew on so many false alarms that they stop listening?"*

Time-series problems hide traps: labels that peek into the future, features that leak, and validation that accidentally trains on tomorrow. This project walks you past each one.

        **What you'll hand in:** a labelled, feature-engineered dataset, a model validated on later days, an alert policy (how many machines to inspect per day), and a business case in rupees.

        **Skills:** time-series features, label engineering, leakage, time-based validation, random forests, precision–recall, alert policies, business case

        **How this works:** each milestone has a task, a code cell for you to finish, and a ✅ check. Hints and a reference solution are hidden until you open them. Try for at least 20 minutes before opening the solution — the struggle is where the learning happens.

## The data

**machine_sensors.csv** — one row per machine per day, 40 machines × 240 days:

| Column | Meaning |
|---|---|
| machine_id | M01 … M40 |
| day | 1 … 240 |
| temperature, vibration, pressure | daily sensor averages |
| failure | 1 on the day the machine broke down |

Machines are repaired the same day and keep running.

In [ ]:
# ▶ Run this cell first: it loads the libraries, the data and the checker.
import math, os, warnings
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
RAW = "https://raw.githubusercontent.com/abpandey-dev/learn-python-from-scratch-to-advanced/main/projects/data/"


def load(name):
    """Read a project data file: from the local data/ folder if it is there, otherwise from GitHub."""
    local = Path("data") / name
    return pd.read_csv(local if local.exists() else RAW + name)


def _plain(x):
    if isinstance(x, pd.DataFrame):
        return {str(c): _plain(x[c].tolist()) for c in x.columns}
    if isinstance(x, pd.Series):
        return {_plain(k): _plain(v) for k, v in x.items()}
    if isinstance(x, np.ndarray):
        return _plain(x.tolist())
    if isinstance(x, np.generic):
        return _plain(x.item())
    if isinstance(x, (list, tuple)):
        return [_plain(v) for v in x]
    if isinstance(x, dict):
        return {_plain(k): _plain(v) for k, v in x.items()}
    return x


def _same(a, b, tol):
    if isinstance(b, bool) or isinstance(a, bool):
        return a == b
    if isinstance(b, (int, float)) and isinstance(a, (int, float)):
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol if tol > 1e-3 else 1e-6)
    if isinstance(b, list) and isinstance(a, list):
        return len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    if isinstance(b, dict) and isinstance(a, dict):
        return set(map(str, a)) == set(map(str, b)) and all(_same(a[k], b[k], tol) for k in a if k in b)
    return a == b


def _why(a, b):
    if a is None:
        return "your function returned None — finish the code and return the result."
    if type(a) != type(b) and not (isinstance(a, (int, float)) and isinstance(b, (int, float))):
        return f"expected a {type(b).__name__}, got a {type(a).__name__}."
    if isinstance(b, (list, dict)) and len(a) != len(b):
        return f"expected {len(b)} items, got {len(a)}."
    if isinstance(b, dict) and set(map(str, a)) != set(map(str, b)):
        return f"the keys should be {sorted(map(str, b))[:8]}."
    return "the values are not right yet. Re-read the task and the hints."


def check(name, fn, want, tol=1e-3):
    """Run your code for one milestone and compare it with the expected answer."""
    try:
        got = _plain(fn())
    except NotImplementedError:
        print(f"⬜ {name}: not started yet.")
        return False
    except Exception as e:
        print(f"❌ {name}: your code raised {type(e).__name__}: {e}")
        return False
    if _same(got, want, tol):
        print(f"✅ {name}: correct!")
        return True
    print(f"❌ {name}: not yet — {_why(got, want)}")
    return False

import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score
sensors = load("machine_sensors.csv").sort_values(["machine_id", "day"]).reset_index(drop=True)
FEATURES = ["temperature", "vibration", "pressure", "vib_mean7", "temp_mean7", "vib_trend"]
print(sensors.shape, "failures:", sensors.failure.sum())


# Used by the checks: your functions chained together.
def prepared():
    return add_features(make_label(sensors))


def test_scores():
    train, test = time_split(prepared())
    model = train_model(train)
    return test.assign(prob=model.predict_proba(test[FEATURES])[:, 1])
print("Ready! Data loaded.")

Look at one machine before modelling. What happens to vibration and temperature before a failure (red lines)?

In [ ]:
m = sensors[sensors.machine_id == "M01"]
fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
ax[0].plot(m.day, m.vibration); ax[0].set_ylabel("vibration")
ax[1].plot(m.day, m.temperature, color="#b45309"); ax[1].set_ylabel("temperature")
for d in m.day[m.failure == 1]:
    for a in ax: a.axvline(d, color="red", alpha=0.5)
plt.show()

## Milestone 1: Build the label: fails in the next 7 days?

We want to predict, at the end of each day, whether the machine will fail **in the next 7 days** (days d+1 … d+7, not today).

Write `make_label(df, horizon=7)` returning a copy of `df` with a `label` column (0/1). Work **within each machine** — one machine's failure must not label another machine's rows.

<details>
<summary><b>💡 Hint</b></summary>

For k in 1..horizon, `df.groupby('machine_id')['failure'].shift(-k)` looks k days ahead on the same machine. A row is positive if any of them is 1 (treat missing as 0).

</details>

In [ ]:
def make_label(df, horizon=7):
    raise NotImplementedError

In [ ]:
check("Build the label: fails in the next 7 days?", lambda: [int(make_label(sensors)['label'].sum()), round(make_label(sensors)['label'].mean(), 4)], [721, 0.0751], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def make_label(df, horizon=7):
    out = df.copy()
    ahead = sum(out.groupby("machine_id")["failure"].shift(-k).fillna(0) for k in range(1, horizon + 1))
    out["label"] = (ahead > 0).astype(int)
    return out
```

</details>

## Milestone 2: Features from the recent past

Write `add_features(df)` adding, per machine:
- `vib_mean7`, `temp_mean7`: mean of the **last 7 days including today** (`min_periods=1`),
- `vib_trend`: vibration today minus vibration 7 days ago (0 when there's no value 7 days ago).

Only use today and earlier — never future rows. The check reads machine M05's features on day 60.

<details>
<summary><b>💡 Hint</b></summary>

`df.groupby('machine_id')['vibration'].transform(lambda s: s.rolling(7, min_periods=1).mean())`.

</details>

In [ ]:
def add_features(df):
    raise NotImplementedError

In [ ]:
check("Features from the recent past", lambda: add_features(sensors).query('machine_id == "M05" and day == 60')[['vib_mean7', 'temp_mean7', 'vib_trend']].round(3).iloc[0].tolist(), [1.1, 62.973, 0.094], tol=1e-06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def add_features(df):
    out = df.copy()
    g = out.groupby("machine_id")
    out["vib_mean7"] = g["vibration"].transform(lambda s: s.rolling(7, min_periods=1).mean())
    out["temp_mean7"] = g["temperature"].transform(lambda s: s.rolling(7, min_periods=1).mean())
    out["vib_trend"] = (out["vibration"] - g["vibration"].shift(7)).fillna(0)
    return out
```

</details>

## Milestone 3: Validate on the future

A random split would let the model learn from day 200 to predict day 150. Instead split by time: **train = days 1–160, test = days 161–240**.

Write `time_split(df)` returning `(train, test)`. The check looks at their sizes and positive rates.

In [ ]:
def time_split(df):
    raise NotImplementedError

In [ ]:
check("Validate on the future", lambda: [len(time_split(prepared())[0]), len(time_split(prepared())[1]), round(time_split(prepared())[0]['label'].mean(), 4), round(time_split(prepared())[1]['label'].mean(), 4)], [6400, 3200, 0.0723, 0.0806], tol=1e-09)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def time_split(df):
    return df[df["day"] <= 160].copy(), df[df["day"] > 160].copy()
```

</details>

## Milestone 4: Train and measure the model

Write `train_model(train)` returning a fitted `RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=0)` trained on `FEATURES` → `label`.

The check reports **average precision** (area under the precision–recall curve) on the test days. With rare positives (~8%) it's far more informative than accuracy: always predicting "no failure" is 92% accurate and useless.

In [ ]:
def train_model(train):
    raise NotImplementedError

In [ ]:
check("Train and measure the model", lambda: round(average_precision_score(test_scores()['label'], test_scores()['prob']), 3), 0.775, tol=0.04)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def train_model(train):
    model = RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=0)
    return model.fit(train[FEATURES], train["label"])
```

</details>

Compare with the base rate: a random ranking would score about the share of positives (≈0.08). How much better is yours?

## Milestone 5: An alert policy the crew can follow

The crew can inspect **3 machines per day**. Each test day, alert the 3 machines with the highest probability.

Write `alerts(scored, k=3)` returning `(number of alerts, precision)` where precision = share of alerts whose `label` is 1, rounded to 3.

<details>
<summary><b>💡 Hint</b></summary>

`scored.sort_values('prob', ascending=False).groupby('day').head(k)`

</details>

In [ ]:
def alerts(scored, k=3):
    raise NotImplementedError

In [ ]:
check("An alert policy the crew can follow", lambda: alerts(test_scores()), [240, 0.596], tol=0.06)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def alerts(scored, k=3):
    chosen = scored.sort_values(["day", "prob"], ascending=[True, False]).groupby("day").head(k)
    return len(chosen), round(chosen["label"].mean(), 3)
```

</details>

## Milestone 6: The business case

A failure is **caught** if its machine got at least one alert in the 7 days before it (days d−7 … d−1). Each caught failure saves ₹2,00,000; each alert costs ₹20,000 for the inspection.

Write `business_case(scored, k=3)` returning `(failures in test, caught, net saving in ₹)`.

Then try k = 1, 2, 3, 5. Which k gives the best net saving?

<details>
<summary><b>💡 Hint</b></summary>

Build the alert set as `{(machine_id, day)}`; for each failure row in the test period, check whether any `(machine, day - j)` for j in 1..7 is in the set.

</details>

In [ ]:
def business_case(scored, k=3):
    raise NotImplementedError

In [ ]:
check("The business case", lambda: business_case(test_scores()), [38, 35, 2200000], tol=0.15)

<details>
<summary><b>🔑 Reference solution (open after you've tried)</b></summary>

```python
def business_case(scored, k=3):
    chosen = scored.sort_values(["day", "prob"], ascending=[True, False]).groupby("day").head(k)
    alerted = set(zip(chosen["machine_id"], chosen["day"]))
    fails = scored[scored["failure"] == 1]
    caught = sum(any((m, d - j) in alerted for j in range(1, 8)) for m, d in zip(fails["machine_id"], fails["day"]))
    return len(fails), int(caught), int(caught * 200000 - len(chosen) * 20000)
```

</details>

The policy is a trade-off you can now show in rupees: more inspections catch more failures but cost more. That chart — net saving vs k — is what convinces a plant manager.

## Present your work

Prepare a one-page note for the plant manager:

1. **Problem**: cost of a breakdown vs an inspection.
2. **What the model sees**: the warning signs (rising vibration and temperature) — include the machine chart.
3. **How we tested it**: trained on days 1–160, tested on 161–240, never on the future.
4. **Policy**: inspect the top k machines per day; failures caught, alerts raised, precision.
5. **Money**: net saving per 80 days, and per year.
6. **Next steps**: more sensors, per-machine baselines, a pilot on one line.

## Put it on your resume

Write bullets with **your own numbers** from the milestones above. A template:

- Built a predictive-maintenance model on 9,600 machine-days of sensor data with leakage-safe labels and rolling features
- Validated on held-out future days (average precision **0.XX** vs a 0.08 base rate)
- Designed a top-k daily inspection policy catching **X of Y** failures, worth an estimated **₹Z** net saving per quarter

Only claim what you built and measured. Put the notebook on your GitHub with a short README: problem, data, approach, results, what you'd do next.

## Interview questions about this project

Practise answering these out loud in 1–2 minutes each:

1. How did you create the label, and what mistake would make it leak the future?
2. Why not a random train/test split here?
3. Why average precision and not accuracy or ROC AUC?
4. How did you choose how many machines to inspect per day?
5. One random vibration spike causes a false alarm. How would you make the model more robust?
6. How would you deploy this so alerts arrive every morning?

## Take it further

- Add features: rolling standard deviation, days since last failure, per-machine z-scores.
- Try gradient boosting and compare average precision on the same time split.
- Plot net saving against k (1–8) and mark the best value.
- Schedule the daily scoring with Airflow (see the Data Engineering course) and send alerts by email.